In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [9]:
X_train = pd.read_csv('../data/X_train_data.csv')
y_train = pd.read_csv('../data/Y_train_data.csv')

X_train = X_train.values
y_train = y_train.values

X_train = torch.from_numpy(X_train).float()
y_train = torch.from_numpy(y_train).float()


In [10]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [11]:
print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)


torch.Size([7864, 84])
torch.Size([7864, 1])
torch.Size([1967, 84])
torch.Size([1967, 1])


In [12]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [13]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

/var/folders/hj/pwp8mnp90wb03qbkc2c60_500000gn/T/ipykernel_1353/2434944468.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
/var/folders/hj/pwp8mnp90wb03qbkc2c60_500000gn/T/ipykernel_1353/2434944468.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_val_tensor = torch.tensor(y_val, dtype=torch.float32)


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import KFold
import itertools

class Net(nn.Module):
    def __init__(self, input_size=84, hidden_layers=[256, 128, 64]):
        super(Net, self).__init__()
        layers = []
        in_features = input_size
        
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(in_features, hidden_size))
            layers.append(nn.ReLU())
            in_features = hidden_size
            
        layers.append(nn.Linear(in_features, 1))
        
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

def train_and_evaluate(model, optimizer, criterion, X_train, y_train, X_val, y_val, epochs=50, batch_size=64):
    num_samples = X_train.shape[0]
    
    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(num_samples)
        
        for i in range(0, num_samples, batch_size):
            indices = permutation[i:i+batch_size]
            batch_x, batch_y = X_train[indices], y_train[indices]
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
    # Validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_mse = torch.mean((val_outputs - y_val)**2).item()
        
    return val_mse

# Hyperparameter Grid
param_grid = {
    'num_layers': [1, 2, 3],
    'num_neurons': [64, 128, 256],
    'epochs': [50, 100, 125, 150]
}

# Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
best_mse = float('inf')
best_params = None
results = []

keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Starting Grid Search with {len(combinations)} combinations...")

for i, params in enumerate(combinations):
    print(f"\nEvaluating config {i+1}/{len(combinations)}: {params}")
    
    fold_mses = []
    
    # K-Fold Loop
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_tensor)):
        # Split data
        X_tr_fold, X_val_fold = X_train_tensor[train_idx], X_train_tensor[val_idx]
        y_tr_fold, y_val_fold = y_train_tensor[train_idx], y_train_tensor[val_idx]
        
        # Prepare hidden layers config
        hidden_layers = [params['num_neurons']] * params['num_layers']
        
        # Initialize model
        model = Net(input_size=84, hidden_layers=hidden_layers)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        
        # Train and Evaluate
        mse = train_and_evaluate(model, optimizer, criterion, X_tr_fold, y_tr_fold, X_val_fold, y_val_fold, epochs=params['epochs'])
        fold_mses.append(mse)
    
    avg_mse = np.mean(fold_mses)
    print(f"Avg MSE: {avg_mse:.4f}")
    
    results.append({'params': params, 'mse': avg_mse})
    
    if avg_mse < best_mse:
        best_mse = avg_mse
        best_params = params

print(f"\nBest Config: {best_params}")
print(f"Best MSE: {best_mse:.4f}")


Starting Grid Search with 36 combinations...

Evaluating config 1/36: {'num_layers': 1, 'num_neurons': 64, 'epochs': 50}
Avg MSE: 0.2669

Evaluating config 2/36: {'num_layers': 1, 'num_neurons': 64, 'epochs': 100}
Avg MSE: 0.2740

Evaluating config 3/36: {'num_layers': 1, 'num_neurons': 64, 'epochs': 125}
Avg MSE: 0.2784

Evaluating config 4/36: {'num_layers': 1, 'num_neurons': 64, 'epochs': 150}
Avg MSE: 0.2847

Evaluating config 5/36: {'num_layers': 1, 'num_neurons': 128, 'epochs': 50}
Avg MSE: 0.2560

Evaluating config 6/36: {'num_layers': 1, 'num_neurons': 128, 'epochs': 100}
Avg MSE: 0.2687

Evaluating config 7/36: {'num_layers': 1, 'num_neurons': 128, 'epochs': 125}
Avg MSE: 0.2705

Evaluating config 8/36: {'num_layers': 1, 'num_neurons': 128, 'epochs': 150}
Avg MSE: 0.2726

Evaluating config 9/36: {'num_layers': 1, 'num_neurons': 256, 'epochs': 50}
Avg MSE: 0.2620

Evaluating config 10/36: {'num_layers': 1, 'num_neurons': 256, 'epochs': 100}
Avg MSE: 0.2561

Evaluating config 11

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split

class OptimalNet(nn.Module):
    def __init__(self):
        super(OptimalNet, self).__init__()
        # Input layer: 84 features
        self.fc1 = nn.Linear(84, 256)
        # Output layer: 1 unit for regression
        self.fc2 = nn.Linear(256, 1)
        

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def train_model():

    net = OptimalNet()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(net.parameters(), lr=0.001)

    # 5. Training Loop
    epochs = 50
    batch_size = 64
    num_samples = X_train_tensor.shape[0]
    
    print("Starting training...")
    for epoch in range(epochs):
        net.train()
        permutation = torch.randperm(num_samples)
        
        train_loss = 0.0
        for i in range(0, num_samples, batch_size):
            indices = permutation[i:i+batch_size]
            batch_x, batch_y = X_train_tensor[indices], y_train_tensor[indices]
            
            optimizer.zero_grad()
            outputs = net(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * batch_x.size(0)
            
        train_loss /= num_samples
        
        # Validation step
        net.eval()
        with torch.no_grad():
            val_outputs = net(X_val_tensor)
            val_loss = criterion(val_outputs, y_val_tensor).item()
            val_mse = torch.mean((val_outputs - y_val_tensor)**2).item()
            val_acc = (torch.abs(val_outputs - y_val_tensor) <= 1.0).float().mean().item() * 100

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val MSE: {val_mse:.4f} | Val Acc: {val_acc:.2f}%")

    return net


In [ ]:
train_model()